In [2]:
import pandas as pd
import numpy as np
import os
import pathlib

p_train, p_test, p_val = 0.8, 0.1, 0.1

xlsx_path = pathlib.Path("../data/solar_panel_data_madagascar.xlsx")


seed = 56

images, elements, coordinates = pd.read_excel(xlsx_path, sheet_name=[0, 1, 2]).values()

FileNotFoundError: [Errno 2] No such file or directory: '../data/solar_panel_data_madagascar.xlsx'

In [3]:
coordinates = coordinates[pd.to_numeric(coordinates["lat"], errors="coerce").notnull()]
coordinates = coordinates[pd.to_numeric(coordinates["long"], errors="coerce").notnull()]
coordinates["lat"] = coordinates["lat"].astype(float)
coordinates["long"] = coordinates["long"].astype(float)
coordinates

,Unnamed: 0,elt_name,edge_rank,long,lat
0,0,2z1,1.0,212.0,116.0
1,1,2z1,2.0,316.0,164.0
2,2,2z1,3.0,284.0,244.0
3,3,2z1,4.0,178.0,205.0
4,4,3z1,1.0,153.0,201.0
...,...,...,...,...,...
97184,97205,2300z24,4.0,3037.0,561.0
97185,97206,2300z25,1.0,3239.0,207.0
97186,97207,2300z25,2.0,3341.0,173.0
97187,97208,2300z25,3.0,3415.0,479.0


In [4]:
elements = elements[elements["type1"] == "pan"]
elements = elements[elements["elt_name"].isin(coordinates["elt_name"])]
elements

,Unnamed: 0,img_name,elt_name,type1,type2,boil_nbr,pan_nbr,pan_area_sqm,notes
0,0,953,953z2,pan,roof,NaN,1.0,0.65,NaN
1,1,691,691z1,pan,roof,NaN,1.0,0.68,NaN
2,2,705,705z2,pan,roof,NaN,1.0,0.68,NaN
3,3,593,593z1,pan,roof,NaN,1.0,0.75,NaN
4,4,593,593z2,pan,roof,NaN,1.0,0.75,NaN
...,...,...,...,...,...,...,...,...,...
22478,22483,2300ID,2300z21,pan,NaN,NaN,36.0,NaN,NaN
22479,22484,2300ID,2300z22,pan,NaN,NaN,36.0,NaN,NaN
22480,22485,2300ID,2300z23,pan,NaN,NaN,36.0,NaN,NaN
22481,22486,2300ID,2300z24,pan,NaN,NaN,36.0,NaN,NaN


In [5]:
images = images[images["img_origin"] == "D"]
images = images[images["type1"] != "boil"]
images = images[images["img_name"].isin(elements["img_name"])]
images

,Unnamed: 0,number,img_name,img_long_East,img_lat_South,img_origin,type1,elt_nbr,type2,park_capacity_kw,...,mouse_alt,width_pixel,height_pixel,city,region,img_date,img_time,img_altsea,zoom,img_ISO
2060,2060,2126,2126ID,43.574062,-23.048845,D,pan,4,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:46:05,49.29,1,100.0
2061,2061,2128,2128ID,43.569401,-23.046642,D,pan,2,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:49:54,83.18,6.3,100.0
2062,2062,2129,2129ID,43.567268,-23.045612,D,mix,7,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:51:26,83.07,2.86,100.0
2063,2063,2130,2130ID,43.567265,-23.045612,D,mix,5,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,06:52:01,83.00,2.86,100.0
2064,2064,2132,2132ID,43.574135,-23.046333,D,pan,20,roof,NaN,...,NaN,5280,3956,Ifaty,South,2023-07-06,07:33:54,73.30,1.26,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11123,11131,11323,11323ID,46.745594,-16.138494,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:57:53,94.53,1.64,100.0
11124,11132,11324,11324ID,46.746655,-16.137222,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:29,90.06,2.58,110.0
11125,11133,11325,11325ID,46.747299,-16.136864,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:45,83.13,1.27,110.0
11126,11134,11326,11326ID,46.747299,-16.136864,D,pan,1,roof,NaN,...,NaN,4000,3000,Amboromalandy,West,2023-09-26,05:58:55,82.66,2.32,100.0


In [6]:
grouped_coordinates = coordinates.groupby("elt_name").agg({"long": list, "lat": list})
bounding_boxes = grouped_coordinates.apply(
    lambda row: [(min(row["long"]) + max(row["long"])) / 2, (min(row["lat"]) + max(row["lat"])) / 2, max(row["long"]) - min(row["long"]), max(row["lat"]) - min(row["lat"])], axis=1
)

In [7]:
bounding_boxes

elt_name
10000z1    [2150.0, 2077.5, 1350.0, 897.0]
10001z1     [2802.5, 2118.0, 687.0, 654.0]
10001z2     [2935.5, 1330.5, 167.0, 183.0]
10002z1    [1598.0, 1634.0, 1230.0, 656.0]
10002z2    [1823.0, 2592.5, 1520.0, 967.0]
                        ...               
999z1           [567.5, 278.0, 57.0, 52.0]
99z1          [223.5, 199.5, 135.0, 123.0]
99z2          [262.0, 386.0, 202.0, 212.0]
9z1            [146.0, 151.0, 118.0, 70.0]
9z2             [172.0, 561.0, 42.0, 40.0]
Length: 22461, dtype: object

In [8]:
grouped_bounding_boxes = bounding_boxes.groupby("elt_name").agg(bounding_boxes=list).reset_index()
elements_with_grouped_bounding_boxes = pd.merge(
    elements, grouped_bounding_boxes, on="elt_name", how="left"
)
elements_with_grouped_bounding_boxes

,Unnamed: 0,img_name,elt_name,type1,type2,boil_nbr,pan_nbr,pan_area_sqm,notes,bounding_boxes
0,0,953,953z2,pan,roof,NaN,1.0,0.65,NaN,"[[241.0, 168.0, 74.0, 78.0]]"
1,1,691,691z1,pan,roof,NaN,1.0,0.68,NaN,"[[208.0, 168.5, 58.0, 61.0]]"
2,2,705,705z2,pan,roof,NaN,1.0,0.68,NaN,"[[387.5, 362.5, 45.0, 61.0]]"
3,3,593,593z1,pan,roof,NaN,1.0,0.75,NaN,"[[93.0, 197.5, 12.0, 33.0]]"
4,4,593,593z2,pan,roof,NaN,1.0,0.75,NaN,"[[119.0, 203.5, 16.0, 33.0]]"
...,...,...,...,...,...,...,...,...,...,...
19960,22483,2300ID,2300z21,pan,NaN,NaN,36.0,NaN,NaN,"[[2200.0, 392.0, 162.0, 406.0]]"
19961,22484,2300ID,2300z22,pan,NaN,NaN,36.0,NaN,NaN,"[[2501.0, 380.0, 120.0, 402.0]]"
19962,22485,2300ID,2300z23,pan,NaN,NaN,36.0,NaN,NaN,"[[2781.0, 380.0, 152.0, 410.0]]"
19963,22486,2300ID,2300z24,pan,NaN,NaN,36.0,NaN,NaN,"[[3055.0, 367.0, 164.0, 388.0]]"


In [9]:
grouped_elements_with_grouped_bounding_boxes = (
    elements_with_grouped_bounding_boxes.groupby("img_name")["bounding_boxes"]
    .agg(list_bounding_boxes="sum")
    .reset_index()
)
images_with_list_bounding_boxes = pd.merge(
    images, grouped_elements_with_grouped_bounding_boxes, on="img_name", how="left"
)
images_with_list_bounding_boxes = images_with_list_bounding_boxes[
    ["img_name", "list_bounding_boxes", "width_pixel",	"height_pixel"]
]
images_with_list_bounding_boxes

,img_name,list_bounding_boxes,width_pixel,height_pixel
0,2126ID,"[[2351.5, 2572.0, 133.0, 108.0], [2376.0, 2634...",5280,3956
1,2128ID,"[[2620.0, 1920.0, 504.0, 416.0], [3028.0, 2008...",5280,3956
2,2129ID,"[[1420.0, 3244.0, 1176.0, 728.0], [1300.0, 260...",5280,3956
3,2130ID,"[[1608.0, 2622.0, 1128.0, 516.0], [1544.0, 281...",5280,3956
4,2132ID,"[[1934.0, 1634.0, 396.0, 92.0], [1886.0, 1776....",5280,3956
...,...,...,...,...
8972,11323ID,"[[1896.0, 1462.0, 328.0, 212.0]]",4000,3000
8973,11324ID,"[[2161.5, 1410.0, 513.0, 132.0]]",4000,3000
8974,11325ID,"[[2409.5, 1578.0, 321.0, 114.0]]",4000,3000
8975,11326ID,"[[1323.0, 1431.0, 258.0, 162.0]]",4000,3000


In [10]:
N = len(images_with_list_bounding_boxes)
N_train = int(p_train * N)
N_test = int(p_test * N)
shuffled_img_names = images_with_list_bounding_boxes.sample(frac=1, random_state=seed).reset_index(
    drop=True
)
df_images = (
    shuffled_img_names[:N_train].reset_index(drop=True),
    shuffled_img_names[N_train : N_train + N_test].reset_index(drop=True),
    shuffled_img_names[N_train + N_test :].reset_index(drop=True),
)

In [11]:
modes = ["train", "test", "val"]

with open(output_path / "solar_panel_dataset.yaml", "w") as f:
    f.write(f"path: {output_path.resolve()}\n")
    for k in range(3):
        f.write(f"{modes[k]}: images / {modes[k]}\n")
    f.write("\nnames:\n\t0: solar_panel")

for k in range(3):
    df = df_images[k]
    mode = modes[k]
    os.makedirs(output_path / "images" / mode, exist_ok=True)
    os.makedirs(output_path / "labels" / mode, exist_ok=True)
    for _, image in df.iterrows():
        with open(output_path / "labels" / mode / (image.img_name + ".txt"), "w") as f:
            for bounding_box in image.list_bounding_boxes:
                x, w = bounding_box[0] / image.width_pixel, bounding_box[2] / image.width_pixel
                y, h = bounding_box[1] / image.height_pixel, bounding_box[3] / image.height_pixel
                f.write(f"0 {x} {y} {w} {h}\n")
            os.rename(image_path / (image.img_name + ".jpg"), output_path / "images" / mode / (image.img_name + ".jpg"))
        